## First trial

In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
import json
import math
import random
import shutil
from typing import Optional, List, Tuple, Dict

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import tqdm

## Configurations

In [2]:
SEED = 123
SOURCE_DIR = Path(r"VOiCES_devkit/source-16k")
WORK_DIR = Path(r"voice_access_files")

# None means the first 5 speakers are allowed
ALLOW_SPEAKERS = None

TRAIN_RATIO = 0.7
VALID_RATIO = 0.1

SEGMENT_SECONDS = 3.0
KEEP_REMAINDER = False

IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 25

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Agregation method
AGGREGATION_METHOD = "mean"   # "mean", "median", "top3mean"

# Augumentation
USE_AUGMENTATION = True

# Experiment name
EXPERIMENT_NAME = "smallcnn_weighted_bce_recording_level"

print("DEVICE =", DEVICE)

DEVICE = cpu


In [3]:
@dataclass
class ProjectPaths:
    work_dir: Path
    recording_meta_path: Path
    segment_meta_path: Path
    spectro_meta_path: Path
    allow_speakers_path: Path
    segment_dir: Path
    spectro_dir: Path
    model_dir: Path
    results_dir: Path

    @classmethod
    def from_work_dir(cls, work_dir: Path | str) -> "ProjectPaths":
        work_dir = Path(work_dir)
        return cls(
            work_dir=work_dir,
            recording_meta_path=work_dir / "metadata_recording_level.csv",
            segment_meta_path=work_dir / "metadata_segment_level.csv",
            spectro_meta_path=work_dir / "metadata_spectrogram_level.csv",
            allow_speakers_path=work_dir / "allow_speakers.json",
            segment_dir=work_dir / "segmented_audio",
            spectro_dir=work_dir / "spectrograms_png",
            model_dir=work_dir / "models",
            results_dir=work_dir / "results",
        )


PATHS = ProjectPaths.from_work_dir(WORK_DIR)


def set_seed(seed: int = 123) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def prepare_dirs(paths: ProjectPaths, clean_intermediate: bool = False) -> None:
    paths.work_dir.mkdir(parents=True, exist_ok=True)
    paths.model_dir.mkdir(parents=True, exist_ok=True)
    paths.results_dir.mkdir(parents=True, exist_ok=True)

    if clean_intermediate:
        for p in [paths.segment_dir, paths.spectro_dir]:
            if p.exists():
                shutil.rmtree(p)

    paths.segment_dir.mkdir(parents=True, exist_ok=True)
    paths.spectro_dir.mkdir(parents=True, exist_ok=True)


set_seed(SEED)
prepare_dirs(PATHS, clean_intermediate=False)

## Additional functions

In [4]:
def load_speaker_to_wavs(source_dir: Path | str) -> Dict[str, List[str]]:
    source_dir = Path(source_dir)
    mapping = {}
    for speaker_dir in sorted(p for p in source_dir.iterdir() if p.is_dir()):
        wavs = sorted(str(p) for p in speaker_dir.rglob("*.wav"))
        if wavs:
            mapping[speaker_dir.name] = wavs
    return mapping


def _split_files_within_speaker(
    files: List[str],
    rng: random.Random,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.1,
) -> Tuple[List[str], List[str], List[str]]:
    files = list(files)
    rng.shuffle(files)
    n = len(files)

    if n == 0:
        return [], [], []
    if n == 1:
        return files, [], []
    if n == 2:
        return [files[0]], [], [files[1]]

    n_train = max(1, int(round(n * train_ratio)))
    n_valid = int(round(n * valid_ratio))

    if n_train >= n:
        n_train = n - 1
    if n_train + n_valid >= n:
        n_valid = max(0, n - n_train - 1)

    if n >= 3 and n_valid == 0:
        n_valid = 1
        if n_train + n_valid >= n:
            n_train = n - n_valid - 1

    train_files = files[:n_train]
    valid_files = files[n_train:n_train + n_valid]
    test_files = files[n_train + n_valid:]

    if len(test_files) == 0:
        test_files = [train_files.pop()]
    return train_files, valid_files, test_files


def _split_speakers(
    speakers: List[str],
    rng: random.Random,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.1,
) -> Tuple[List[str], List[str], List[str]]:
    speakers = list(speakers)
    rng.shuffle(speakers)
    n = len(speakers)

    if n == 0:
        return [], [], []
    if n == 1:
        return speakers, [], []
    if n == 2:
        return [speakers[0]], [], [speakers[1]]

    n_train = max(1, int(round(n * train_ratio)))
    n_valid = int(round(n * valid_ratio))

    if n_train >= n:
        n_train = n - 1
    if n_train + n_valid >= n:
        n_valid = max(0, n - n_train - 1)

    if n >= 3 and n_valid == 0:
        n_valid = 1
        if n_train + n_valid >= n:
            n_train = n - n_valid - 1

    train_speakers = speakers[:n_train]
    valid_speakers = speakers[n_train:n_train + n_valid]
    test_speakers = speakers[n_train + n_valid:]

    if len(test_speakers) == 0:
        test_speakers = [train_speakers.pop()]
    return train_speakers, valid_speakers, test_speakers

## Building recording-level metadata

In [5]:
def build_recording_metadata(
    source_dir: Path | str,
    paths: ProjectPaths,
    allow_speakers: Optional[List[str]] = None,
    seed: int = 123,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.1,
) -> pd.DataFrame:
    source_dir = Path(source_dir)
    rng = random.Random(seed)

    speaker_to_wavs = load_speaker_to_wavs(source_dir)
    all_speakers = sorted(speaker_to_wavs.keys())

    if allow_speakers is None:
        allow_speakers = all_speakers[:5]

    allow_speakers = sorted(allow_speakers)
    not_allow_speakers = [s for s in all_speakers if s not in allow_speakers]

    rows = []

    # class 1 = allow
    for speaker in allow_speakers:
        wavs = speaker_to_wavs[speaker]
        train_wavs, valid_wavs, test_wavs = _split_files_within_speaker(
            wavs,
            rng=rng,
            train_ratio=train_ratio,
            valid_ratio=valid_ratio,
        )
        for split, split_wavs in [("train", train_wavs), ("valid", valid_wavs), ("test", test_wavs)]:
            for audio_path in split_wavs:
                rel = Path(audio_path).relative_to(source_dir)
                recording_id = str(rel).replace("\\", "__").replace("/", "__").replace(".wav", "")
                rows.append(
                    {
                        "recording_id": recording_id,
                        "speaker": speaker,
                        "label": 1,
                        "class_name": "allow",
                        "split": split,
                        "audio_path": str(audio_path),
                    }
                )

    # class 0 = not_allow, speaker-disjoint
    train_speakers, valid_speakers, test_speakers = _split_speakers(
        not_allow_speakers,
        rng=rng,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
    )
    speaker_to_split = {}
    for s in train_speakers:
        speaker_to_split[s] = "train"
    for s in valid_speakers:
        speaker_to_split[s] = "valid"
    for s in test_speakers:
        speaker_to_split[s] = "test"

    for speaker in not_allow_speakers:
        split = speaker_to_split[speaker]
        for audio_path in speaker_to_wavs[speaker]:
            rel = Path(audio_path).relative_to(source_dir)
            recording_id = str(rel).replace("\\", "__").replace("/", "__").replace(".wav", "")
            rows.append(
                {
                    "recording_id": recording_id,
                    "speaker": speaker,
                    "label": 0,
                    "class_name": "not_allow",
                    "split": split,
                    "audio_path": str(audio_path),
                }
            )

    meta = pd.DataFrame(rows).sort_values(["split", "label", "speaker", "audio_path"]).reset_index(drop=True)
    meta.to_csv(paths.recording_meta_path, index=False)

    with open(paths.allow_speakers_path, "w", encoding="utf-8") as f:
        json.dump({"allow_speakers": allow_speakers}, f, ensure_ascii=False, indent=2)

    return meta


def validate_recording_metadata(meta: pd.DataFrame) -> None:
    assert set(meta["label"].unique()) <= {0, 1}
    assert set(meta["split"].unique()) <= {"train", "valid", "test"}

    dup = meta.duplicated(subset=["audio_path"]).sum()
    assert dup == 0, f"Duplicate audio_path rows found: {dup}"

    # twardy brak przecieku tego samego nagrania miedzy splitami
    split_counts = meta.groupby("audio_path")["split"].nunique()
    assert int((split_counts > 1).sum()) == 0, "Same audio_path appears in multiple splits"

    # not_allow speaker-disjoint
    not_allow = meta[meta["label"] == 0].copy()
    speaker_split_counts = not_allow.groupby("speaker")["split"].nunique()
    assert int((speaker_split_counts > 1).sum()) == 0, "not_allow speaker appears in multiple splits"

    print("Recording metadata OK")
    print(meta.groupby(["split", "class_name"]).size().unstack(fill_value=0))
    print()
    print("Unique speakers:")
    print(meta.groupby(["split", "class_name"])["speaker"].nunique().unstack(fill_value=0))

In [6]:
recording_meta = build_recording_metadata(
    source_dir=SOURCE_DIR,
    paths=PATHS,
    allow_speakers=ALLOW_SPEAKERS,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    valid_ratio=VALID_RATIO,
)

validate_recording_metadata(recording_meta)
recording_meta.head()

Recording metadata OK
class_name  allow  not_allow
split                       
test            5        118
train           5        412
valid           0         60

Unique speakers:
class_name  allow  not_allow
split                       
test            5         59
train           5        206
valid           0         30


,recording_id,speaker,label,class_name,split,audio_path
0,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...
1,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch124588-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...
2,sp0154__Lab41-SRI-VOiCES-src-sp0154-ch123998-s...,sp0154,0,not_allow,test,VOiCES_devkit\source-16k\sp0154\Lab41-SRI-VOiC...
3,sp0154__Lab41-SRI-VOiCES-src-sp0154-ch124002-s...,sp0154,0,not_allow,test,VOiCES_devkit\source-16k\sp0154\Lab41-SRI-VOiC...
4,sp0174__Lab41-SRI-VOiCES-src-sp0174-ch050561-s...,sp0174,0,not_allow,test,VOiCES_devkit\source-16k\sp0174\Lab41-SRI-VOiC...


## Audio segmentation

In [7]:
def _pad_or_trim_segment(y: np.ndarray, target_len: int) -> np.ndarray:
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    elif len(y) > target_len:
        y = y[:target_len]
    return y


def segment_recordings(
    paths: ProjectPaths,
    segment_seconds: float = 3.0,
    keep_remainder: bool = False,
) -> pd.DataFrame:
    rec_meta = pd.read_csv(paths.recording_meta_path)
    rows = []

    for row in rec_meta.itertuples(index=False):
        y, sr = librosa.load(row.audio_path, sr=None, mono=True)
        segment_len = int(round(segment_seconds * sr))

        if len(y) <= segment_len:
            seg = _pad_or_trim_segment(y, segment_len)
            seg_idx = 0
            segment_name = f"{row.recording_id}__seg{seg_idx:04d}.wav"
            out_path = paths.segment_dir / row.split / row.class_name / row.speaker / segment_name
            out_path.parent.mkdir(parents=True, exist_ok=True)
            sf.write(out_path, seg, sr)
            rows.append(
                {
                    "recording_id": row.recording_id,
                    "speaker": row.speaker,
                    "label": int(row.label),
                    "class_name": row.class_name,
                    "split": row.split,
                    "audio_path": row.audio_path,
                    "segment_index": seg_idx,
                    "segment_path": str(out_path),
                    "sr": sr,
                    "segment_seconds": segment_seconds,
                }
            )
            continue

        seg_idx = 0
        for start in range(0, len(y), segment_len):
            end = start + segment_len
            seg = y[start:end]

            if len(seg) < segment_len:
                if not keep_remainder:
                    break
                seg = _pad_or_trim_segment(seg, segment_len)

            segment_name = f"{row.recording_id}__seg{seg_idx:04d}.wav"
            out_path = paths.segment_dir / row.split / row.class_name / row.speaker / segment_name
            out_path.parent.mkdir(parents=True, exist_ok=True)
            sf.write(out_path, seg, sr)

            rows.append(
                {
                    "recording_id": row.recording_id,
                    "speaker": row.speaker,
                    "label": int(row.label),
                    "class_name": row.class_name,
                    "split": row.split,
                    "audio_path": row.audio_path,
                    "segment_index": seg_idx,
                    "segment_path": str(out_path),
                    "sr": sr,
                    "segment_seconds": segment_seconds,
                }
            )
            seg_idx += 1

    seg_meta = pd.DataFrame(rows).sort_values(["split", "label", "speaker", "recording_id", "segment_index"]).reset_index(drop=True)
    seg_meta.to_csv(paths.segment_meta_path, index=False)
    return seg_meta


def validate_segment_metadata(seg_meta: pd.DataFrame) -> None:
    assert set(seg_meta["label"].unique()) <= {0, 1}
    assert set(seg_meta["split"].unique()) <= {"train", "valid", "test"}

    # record only in one set
    dup = seg_meta.duplicated(subset=["segment_path"]).sum()
    assert dup == 0, f"Duplicate segment_path rows found: {dup}"

    split_counts = seg_meta.groupby("recording_id")["split"].nunique()
    assert int((split_counts > 1).sum()) == 0, "recording_id appears in multiple splits"

    print("Segment metadata OK")
    print(seg_meta.groupby(["split", "class_name"]).size().unstack(fill_value=0))

In [8]:
segment_meta = segment_recordings(
    paths=PATHS,
    segment_seconds=SEGMENT_SECONDS,
    keep_remainder=KEEP_REMAINDER,
)

validate_segment_metadata(segment_meta)
segment_meta.head()

C:\Users\idani\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Segment metadata OK
class_name  allow  not_allow
split                       
test           25        578
train          25       1996
valid           0        291


,recording_id,speaker,label,class_name,split,audio_path,segment_index,segment_path,sr,segment_seconds
0,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,0,voice_access_files\segmented_audio\test\not_al...,16000,3.0
1,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,1,voice_access_files\segmented_audio\test\not_al...,16000,3.0
2,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,2,voice_access_files\segmented_audio\test\not_al...,16000,3.0
3,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,3,voice_access_files\segmented_audio\test\not_al...,16000,3.0
4,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,4,voice_access_files\segmented_audio\test\not_al...,16000,3.0


## Mel-spectograms PNG

In [9]:
def mel_spectrogram_uint8(
    y: np.ndarray,
    sr: int,
    image_size: int = 128,
    n_mels: int = 128,
    n_fft: int = 1024,
    hop_length: int = 256,
) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # skala do [0, 255]
    mel_db = mel_db.astype(np.float32)
    mel_db -= mel_db.min()
    mel_db /= max(mel_db.max(), 1e-8)
    arr = (255.0 * mel_db).clip(0, 255).astype(np.uint8)

    img = Image.fromarray(arr, mode="L").resize((image_size, image_size), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


def build_spectrogram_pngs(paths: ProjectPaths, image_size: int = 128) -> pd.DataFrame:
    seg_meta = pd.read_csv(paths.segment_meta_path)
    rows = []

    for row in seg_meta.itertuples(index=False):
        y, sr = librosa.load(row.segment_path, sr=None, mono=True)
        arr = mel_spectrogram_uint8(y, sr=sr, image_size=image_size)

        png_name = Path(row.segment_path).with_suffix(".png").name
        out_path = paths.spectro_dir / row.split / row.class_name / row.speaker / png_name
        out_path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(arr, mode="L").save(out_path)

        rows.append(
            {
                "recording_id": row.recording_id,
                "speaker": row.speaker,
                "label": int(row.label),
                "class_name": row.class_name,
                "split": row.split,
                "audio_path": row.audio_path,
                "segment_index": int(row.segment_index),
                "segment_path": row.segment_path,
                "spectrogram_path": str(out_path),
            }
        )

    spec_meta = pd.DataFrame(rows).sort_values(["split", "label", "speaker", "recording_id", "segment_index"]).reset_index(drop=True)
    spec_meta.to_csv(paths.spectro_meta_path, index=False)
    return spec_meta

In [10]:
spectro_meta = build_spectrogram_pngs(PATHS, image_size=IMAGE_SIZE)
spectro_meta.head()

,recording_id,speaker,label,class_name,split,audio_path,segment_index,segment_path,spectrogram_path
0,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,0,voice_access_files\segmented_audio\test\not_al...,voice_access_files\spectrograms_png\test\not_a...
1,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,1,voice_access_files\segmented_audio\test\not_al...,voice_access_files\spectrograms_png\test\not_a...
2,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,2,voice_access_files\segmented_audio\test\not_al...,voice_access_files\spectrograms_png\test\not_a...
3,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,3,voice_access_files\segmented_audio\test\not_al...,voice_access_files\spectrograms_png\test\not_a...
4,sp0118__Lab41-SRI-VOiCES-src-sp0118-ch121721-s...,sp0118,0,not_allow,test,VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...,4,voice_access_files\segmented_audio\test\not_al...,voice_access_files\spectrograms_png\test\not_a...


## Dataset, augumentation i DataLoadery

In [11]:
def apply_specaugment_np(
    arr: np.ndarray,
    max_time_masks: int = 2,
    max_freq_masks: int = 2,
    time_mask_fraction: float = 0.10,
    freq_mask_fraction: float = 0.10,
) -> np.ndarray:
    arr = arr.copy()
    h, w = arr.shape

    n_time_masks = np.random.randint(0, max_time_masks + 1)
    for _ in range(n_time_masks):
        width = np.random.randint(1, max(2, int(w * time_mask_fraction)))
        start = np.random.randint(0, max(1, w - width + 1))
        arr[:, start:start + width] = 0.0

    n_freq_masks = np.random.randint(0, max_freq_masks + 1)
    for _ in range(n_freq_masks):
        height = np.random.randint(1, max(2, int(h * freq_mask_fraction)))
        start = np.random.randint(0, max(1, h - height + 1))
        arr[start:start + height, :] = 0.0

    return arr


def load_png_as_tensor(
    image_path: str | Path,
    image_size: int = 128,
    mean: Optional[float] = None,
    std: Optional[float] = None,
    augment: bool = False,
) -> torch.Tensor:
    img = Image.open(image_path).convert("L").resize((image_size, image_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0

    if augment:
        arr = apply_specaugment_np(arr)

    x = torch.from_numpy(arr).unsqueeze(0)
    if mean is not None and std is not None:
        x = (x - float(mean)) / max(float(std), 1e-8)
    return x.float()


class SpectrogramDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_size: int = 128,
        mean: Optional[float] = None,
        std: Optional[float] = None,
        augment: bool = False,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.image_size = image_size
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        x = load_png_as_tensor(
            row["spectrogram_path"],
            image_size=self.image_size,
            mean=self.mean,
            std=self.std,
            augment=self.augment,
        )
        y = torch.tensor(float(row["label"]), dtype=torch.float32)
        return x, y, row["recording_id"], row["speaker"], row["audio_path"]


def compute_train_mean_std(train_df: pd.DataFrame, image_size: int = 128) -> Tuple[float, float]:
    sum_val = 0.0
    sum_sq = 0.0
    n_pix = 0

    for image_path in train_df["spectrogram_path"]:
        img = Image.open(image_path).convert("L").resize((image_size, image_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        sum_val += float(arr.sum())
        sum_sq += float((arr ** 2).sum())
        n_pix += arr.size

    mean = sum_val / max(n_pix, 1)
    var = sum_sq / max(n_pix, 1) - mean ** 2
    std = math.sqrt(max(var, 1e-12))
    return mean, std


def make_loaders(
    paths: ProjectPaths,
    image_size: int = 128,
    batch_size: int = 32,
    augment: bool = True,
):
    meta = pd.read_csv(paths.spectro_meta_path)

    train_df = meta[meta["split"] == "train"].copy()
    valid_df = meta[meta["split"] == "valid"].copy()
    test_df = meta[meta["split"] == "test"].copy()

    train_mean, train_std = compute_train_mean_std(train_df, image_size=image_size)

    train_ds = SpectrogramDataset(train_df, image_size=image_size, mean=train_mean, std=train_std, augment=augment)
    train_eval_ds = SpectrogramDataset(train_df, image_size=image_size, mean=train_mean, std=train_std, augment=False)
    valid_ds = SpectrogramDataset(valid_df, image_size=image_size, mean=train_mean, std=train_std, augment=False)
    test_ds = SpectrogramDataset(test_df, image_size=image_size, mean=train_mean, std=train_std, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    stats = {
        "train_mean": float(train_mean),
        "train_std": float(train_std),
        "n_train_segments": int(len(train_df)),
        "n_valid_segments": int(len(valid_df)),
        "n_test_segments": int(len(test_df)),
        "n_train_recordings": int(train_df["recording_id"].nunique()),
        "n_valid_recordings": int(valid_df["recording_id"].nunique()),
        "n_test_recordings": int(test_df["recording_id"].nunique()),
    }
    return train_loader, train_eval_loader, valid_loader, test_loader, stats

## Model, metrics and training

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, use_batchnorm: bool = True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        ]
        if use_batchnorm:
            layers.insert(1, nn.BatchNorm2d(out_ch))
        layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class SmallCNN(nn.Module):
    def __init__(self, dropout: float = 0.3, use_batchnorm: bool = True):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 16, use_batchnorm=use_batchnorm),
            ConvBlock(16, 32, use_batchnorm=use_batchnorm),
            ConvBlock(32, 64, use_batchnorm=use_batchnorm),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x.squeeze(1)


def collect_segment_predictions(model: nn.Module, loader: DataLoader, device: str) -> pd.DataFrame:
    model.eval()
    rows = []

    with torch.no_grad():
        for x, y, recording_ids, speakers, audio_paths in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            y_np = y.numpy()

            for i in range(len(probs)):
                rows.append(
                    {
                        "recording_id": recording_ids[i],
                        "speaker": speakers[i],
                        "audio_path": audio_paths[i],
                        "y_true": int(y_np[i]),
                        "p_allow_segment": float(probs[i]),
                    }
                )

    return pd.DataFrame(rows)


def aggregate_recording_predictions(
    segment_df: pd.DataFrame,
    method: str = "mean",
    top_k: int = 3,
) -> pd.DataFrame:
    if len(segment_df) == 0:
        return pd.DataFrame(columns=["recording_id", "speaker", "audio_path", "y_true", "p_allow"])

    rows = []
    for recording_id, g in segment_df.groupby("recording_id"):
        probs = g["p_allow_segment"].to_numpy(dtype=float)

        if method == "mean":
            p_allow = float(np.mean(probs))
        elif method == "median":
            p_allow = float(np.median(probs))
        elif method == "top3mean":
            k = min(top_k, len(probs))
            p_allow = float(np.mean(np.sort(probs)[-k:]))
        else:
            raise ValueError(f"Unknown aggregation method: {method}")

        rows.append(
            {
                "recording_id": recording_id,
                "speaker": g["speaker"].iloc[0],
                "audio_path": g["audio_path"].iloc[0],
                "y_true": int(g["y_true"].iloc[0]),
                "p_allow": p_allow,
                "n_segments": int(len(g)),
            }
        )

    return pd.DataFrame(rows).sort_values(["y_true", "speaker", "recording_id"]).reset_index(drop=True)


def binary_metrics_from_probs(y_true: np.ndarray, p_allow: np.ndarray, threshold: float) -> dict:
    y_true = np.asarray(y_true).astype(int)
    p_allow = np.asarray(p_allow).astype(float)
    y_pred = (p_allow >= threshold).astype(int)

    n_total = len(y_true)
    n_allow = int((y_true == 1).sum())
    n_not_allow = int((y_true == 0).sum())

    false_accept = int(((y_true == 0) & (y_pred == 1)).sum())
    false_reject = int(((y_true == 1) & (y_pred == 0)).sum())
    correct = int((y_true == y_pred).sum())

    acc = correct / max(n_total, 1)
    far = false_accept / max(n_not_allow, 1)
    frr = false_reject / max(n_allow, 1)
    balanced_error = 0.5 * (far + frr)
    gap = abs(far - frr)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    return {
        "acc": float(acc),
        "far": float(far),
        "frr": float(frr),
        "balanced_error": float(balanced_error),
        "gap_far_frr": float(gap),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "n_total": n_total,
        "n_allow": n_allow,
        "n_not_allow": n_not_allow,
    }


def find_best_threshold(
    y_true: np.ndarray,
    p_allow: np.ndarray,
    penalty_gap: float = 0.25,
) -> dict:
    rows = []
    for t in np.linspace(0.01, 0.99, 99):
        m = binary_metrics_from_probs(y_true, p_allow, threshold=float(t))
        score = m["balanced_error"] + penalty_gap * m["gap_far_frr"]
        rows.append({"threshold": float(t), "score": float(score), **m})

    th_df = pd.DataFrame(rows).sort_values(["score", "balanced_error", "gap_far_frr", "threshold"]).reset_index(drop=True)
    return th_df.iloc[0].to_dict()


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    threshold: float,
    aggregation_method: str = "mean",
) -> dict:
    seg_df = collect_segment_predictions(model, loader, device=device)
    rec_df = aggregate_recording_predictions(seg_df, method=aggregation_method)

    segment_metrics = binary_metrics_from_probs(
        y_true=seg_df["y_true"].to_numpy(),
        p_allow=seg_df["p_allow_segment"].to_numpy(),
        threshold=threshold,
    )
    recording_metrics = binary_metrics_from_probs(
        y_true=rec_df["y_true"].to_numpy(),
        p_allow=rec_df["p_allow"].to_numpy(),
        threshold=threshold,
    )

    return {
        "segment_df": seg_df,
        "recording_df": rec_df,
        "segment_metrics": segment_metrics,
        "recording_metrics": recording_metrics,
    }


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    train_eval_loader: DataLoader,
    valid_loader: DataLoader,
    device: str,
    epochs: int = 25,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    aggregation_method: str = "mean",
    checkpoint_path: Optional[Path] = None,
    train_mean: Optional[float] = None,
    train_std: Optional[float] = None,
    image_size: Optional[int] = None,
    model_config: Optional[dict] = None,
) -> pd.DataFrame:
    model = model.to(device)

    train_targets = []
    for _, y, *_ in train_loader:
        train_targets.extend(y.tolist())
    train_targets = np.asarray(train_targets, dtype=np.float32)

    n_pos = float((train_targets == 1).sum())
    n_neg = float((train_targets == 0).sum())
    pos_weight_value = n_neg / max(n_pos, 1.0)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history_rows = []
    best_score = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        n_seen = 0

        for x, y, *_ in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * x.size(0)
            n_seen += x.size(0)

        train_loss = loss_sum / max(n_seen, 1)

        valid_seg_df = collect_segment_predictions(model, valid_loader, device=device)
        valid_rec_df = aggregate_recording_predictions(valid_seg_df, method=aggregation_method)
        best_thr = find_best_threshold(
            y_true=valid_rec_df["y_true"].to_numpy(),
            p_allow=valid_rec_df["p_allow"].to_numpy(),
        )

        train_eval = evaluate_model(
            model=model,
            loader=train_eval_loader,
            device=device,
            threshold=best_thr["threshold"],
            aggregation_method=aggregation_method,
        )
        valid_eval = evaluate_model(
            model=model,
            loader=valid_loader,
            device=device,
            threshold=best_thr["threshold"],
            aggregation_method=aggregation_method,
        )

        score = valid_eval["recording_metrics"]["balanced_error"] + 0.25 * valid_eval["recording_metrics"]["gap_far_frr"]

        if score < best_score:
            best_score = score
            if checkpoint_path is not None:
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "threshold": float(best_thr["threshold"]),
                        "aggregation_method": aggregation_method,
                        "train_mean": float(train_mean),
                        "train_std": float(train_std),
                        "image_size": int(image_size),
                        "model_config": model_config or {},
                    },
                    checkpoint_path,
                )

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": float(train_loss),
                "pos_weight": float(pos_weight_value),
                "threshold": float(best_thr["threshold"]),
                "train_acc": train_eval["recording_metrics"]["acc"],
                "train_far": train_eval["recording_metrics"]["far"],
                "train_frr": train_eval["recording_metrics"]["frr"],
                "train_balanced_error": train_eval["recording_metrics"]["balanced_error"],
                "valid_acc": valid_eval["recording_metrics"]["acc"],
                "valid_far": valid_eval["recording_metrics"]["far"],
                "valid_frr": valid_eval["recording_metrics"]["frr"],
                "valid_balanced_error": valid_eval["recording_metrics"]["balanced_error"],
                "valid_gap_far_frr": valid_eval["recording_metrics"]["gap_far_frr"],
                "valid_score": float(score),
            }
        )

    return pd.DataFrame(history_rows)

## One run

In [13]:
def run_experiment(
    paths: ProjectPaths,
    experiment_name: str,
    image_size: int = 128,
    batch_size: int = 32,
    epochs: int = 25,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    dropout: float = 0.3,
    use_batchnorm: bool = True,
    augment: bool = True,
    aggregation_method: str = "mean",
    device: str = "cpu",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_loader, train_eval_loader, valid_loader, test_loader, stats = make_loaders(
        paths=paths,
        image_size=image_size,
        batch_size=batch_size,
        augment=augment,
    )

    model = SmallCNN(dropout=dropout, use_batchnorm=use_batchnorm)

    checkpoint_path = paths.model_dir / f"{experiment_name}.pt"
    history = train_model(
        model=model,
        train_loader=train_loader,
        train_eval_loader=train_eval_loader,
        valid_loader=valid_loader,
        device=device,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        aggregation_method=aggregation_method,
        checkpoint_path=checkpoint_path,
        train_mean=stats["train_mean"],
        train_std=stats["train_std"],
        image_size=image_size,
        model_config={
            "dropout": dropout,
            "use_batchnorm": use_batchnorm,
        },
    )

    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device)
    threshold = float(ckpt["threshold"])

    train_eval = evaluate_model(model, train_eval_loader, device=device, threshold=threshold, aggregation_method=aggregation_method)
    valid_eval = evaluate_model(model, valid_loader, device=device, threshold=threshold, aggregation_method=aggregation_method)
    test_eval = evaluate_model(model, test_loader, device=device, threshold=threshold, aggregation_method=aggregation_method)

    results = pd.DataFrame(
        [
            {
                "experiment_name": experiment_name,
                "threshold": threshold,
                "aggregation_method": aggregation_method,
                "dropout": dropout,
                "use_batchnorm": use_batchnorm,
                "augment": augment,
                "epochs": epochs,
                "batch_size": batch_size,
                "lr": lr,
                "weight_decay": weight_decay,
                "train_far": train_eval["recording_metrics"]["far"],
                "train_frr": train_eval["recording_metrics"]["frr"],
                "train_acc": train_eval["recording_metrics"]["acc"],
                "train_balanced_error": train_eval["recording_metrics"]["balanced_error"],
                "valid_far": valid_eval["recording_metrics"]["far"],
                "valid_frr": valid_eval["recording_metrics"]["frr"],
                "valid_acc": valid_eval["recording_metrics"]["acc"],
                "valid_balanced_error": valid_eval["recording_metrics"]["balanced_error"],
                "test_far": test_eval["recording_metrics"]["far"],
                "test_frr": test_eval["recording_metrics"]["frr"],
                "test_acc": test_eval["recording_metrics"]["acc"],
                "test_balanced_error": test_eval["recording_metrics"]["balanced_error"],
                "test_gap_far_frr": test_eval["recording_metrics"]["gap_far_frr"],
                "test_segment_far": test_eval["segment_metrics"]["far"],
                "test_segment_frr": test_eval["segment_metrics"]["frr"],
                "test_segment_acc": test_eval["segment_metrics"]["acc"],
                **stats,
            }
        ]
    )

    history.to_csv(paths.results_dir / f"{experiment_name}_history.csv", index=False)
    results.to_csv(paths.results_dir / f"{experiment_name}_results.csv", index=False)
    train_eval["recording_df"].to_csv(paths.results_dir / f"{experiment_name}_train_recording_predictions.csv", index=False)
    valid_eval["recording_df"].to_csv(paths.results_dir / f"{experiment_name}_valid_recording_predictions.csv", index=False)
    test_eval["recording_df"].to_csv(paths.results_dir / f"{experiment_name}_test_recording_predictions.csv", index=False)

    return history, results

In [14]:
history_df, results_df = run_experiment(
    paths=PATHS,
    experiment_name=EXPERIMENT_NAME,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    dropout=0.3,
    use_batchnorm=True,
    augment=USE_AUGMENTATION,
    aggregation_method=AGGREGATION_METHOD,
    device=DEVICE,
)

print("Train history:")
display(history_df.tail())

print("Final result (recording-level FAR/FRR):")
display(results_df)

Historia treningu:


,epoch,train_loss,pos_weight,threshold,train_acc,train_far,train_frr,train_balanced_error,valid_acc,valid_far,valid_frr,valid_balanced_error,valid_gap_far_frr,valid_score
20,21,0.871922,79.84,0.97,0.985612,0.002427,1.0,0.501214,1.0,0.0,0.0,0.0,0.0,0.0
21,22,0.809830,79.84,0.80,0.985612,0.004854,0.8,0.402427,1.0,0.0,0.0,0.0,0.0,0.0
22,23,0.713106,79.84,0.52,0.973621,0.024272,0.2,0.112136,1.0,0.0,0.0,0.0,0.0,0.0
23,24,0.673309,79.84,0.87,0.990408,0.002427,0.6,0.301214,1.0,0.0,0.0,0.0,0.0,0.0
24,25,0.657722,79.84,0.89,0.990408,0.002427,0.6,0.301214,1.0,0.0,0.0,0.0,0.0,0.0


Wynik koncowy (recording-level FAR/FRR):


,experiment_name,threshold,aggregation_method,dropout,use_batchnorm,augment,epochs,batch_size,lr,weight_decay,...,test_segment_frr,test_segment_acc,train_mean,train_std,n_train_segments,n_valid_segments,n_test_segments,n_train_recordings,n_valid_recordings,n_test_recordings
0,smallcnn_weighted_bce_recording_level,0.52,mean,0.3,True,True,25,32,0.001,0.0001,...,0.88,0.956882,0.33879,0.212687,2021,291,603,417,60,123


## Some experiments

In [15]:
EXPERIMENT_CONFIGS = [
    {
        "experiment_name": "exp_base",
        "dropout": 0.3,
        "use_batchnorm": True,
        "augment": True,
        "aggregation_method": "mean",
    },
    {
        "experiment_name": "exp_no_augment",
        "dropout": 0.3,
        "use_batchnorm": True,
        "augment": False,
        "aggregation_method": "mean",
    },
    {
        "experiment_name": "exp_more_dropout",
        "dropout": 0.5,
        "use_batchnorm": True,
        "augment": True,
        "aggregation_method": "mean",
    },
    {
        "experiment_name": "exp_no_batchnorm",
        "dropout": 0.3,
        "use_batchnorm": False,
        "augment": True,
        "aggregation_method": "mean",
    },
    {
        "experiment_name": "exp_median_aggregation",
        "dropout": 0.3,
        "use_batchnorm": True,
        "augment": True,
        "aggregation_method": "median",
    },
]

all_results = []
for cfg in EXPERIMENT_CONFIGS:
    _, res = run_experiment(
        paths=PATHS,
        experiment_name=cfg["experiment_name"],
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        lr=1e-3,
        weight_decay=1e-4,
        dropout=cfg["dropout"],
        use_batchnorm=cfg["use_batchnorm"],
        augment=cfg["augment"],
        aggregation_method=cfg["aggregation_method"],
        device=DEVICE,
    )
    all_results.append(res)

comparison_df = pd.concat(all_results, ignore_index=True)
comparison_df = comparison_df.sort_values(
    ["test_balanced_error", "test_gap_far_frr", "valid_balanced_error", "experiment_name"]
).reset_index(drop=True)

display(comparison_df[[
    "experiment_name",
    "threshold",
    "aggregation_method",
    "test_far",
    "test_frr",
    "test_acc",
    "test_balanced_error",
    "test_gap_far_frr",
]])

KeyboardInterrupt: 

## Append new speaker

In [ ]:
def append_new_allow_speaker(
    paths: ProjectPaths,
    new_person_dir: Path | str,
    seed: int = 123,
    train_ratio: float = 0.7,
    valid_ratio: float = 0.1,
) -> pd.DataFrame:
    rng = random.Random(seed)
    new_person_dir = Path(new_person_dir)

    wavs = sorted(str(p) for p in new_person_dir.rglob("*.wav"))
    if not wavs:
        raise ValueError(f"No wav files found in {new_person_dir}")

    if paths.recording_meta_path.exists():
        meta = pd.read_csv(paths.recording_meta_path)
    else:
        raise ValueError("No recording metadata found. Build the project first.")

    speaker = new_person_dir.name
    meta = meta[meta["speaker"] != speaker].copy()

    train_wavs, valid_wavs, test_wavs = _split_files_within_speaker(
        wavs,
        rng=rng,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
    )

    new_rows = []
    for split, split_wavs in [("train", train_wavs), ("valid", valid_wavs), ("test", test_wavs)]:
        for audio_path in split_wavs:
            recording_id = f"{speaker}__{Path(audio_path).stem}"
            new_rows.append(
                {
                    "recording_id": recording_id,
                    "speaker": speaker,
                    "label": 1,
                    "class_name": "allow",
                    "split": split,
                    "audio_path": str(audio_path),
                }
            )

    updated = (
        pd.concat([meta, pd.DataFrame(new_rows)], ignore_index=True)
        .sort_values(["split", "label", "speaker", "audio_path"])
        .reset_index(drop=True)
    )
    updated.to_csv(paths.recording_meta_path, index=False)

    if paths.allow_speakers_path.exists():
        with open(paths.allow_speakers_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        current_allow = sorted(set(payload.get("allow_speakers", [])) | {speaker})
    else:
        current_allow = [speaker]

    with open(paths.allow_speakers_path, "w", encoding="utf-8") as f:
        json.dump({"allow_speakers": current_allow}, f, ensure_ascii=False, indent=2)

    return updated


def rebuild_after_new_person(
    paths: ProjectPaths,
    segment_seconds: float = 3.0,
    keep_remainder: bool = False,
    image_size: int = 128,
):
    segment_meta = segment_recordings(paths, segment_seconds=segment_seconds, keep_remainder=keep_remainder)
    spectro_meta = build_spectrogram_pngs(paths, image_size=image_size)
    return segment_meta, spectro_meta

## Prediction for new recording

In [ ]:
def load_checkpoint_and_model(checkpoint_path: Path | str, device: str = "cpu") -> Tuple[nn.Module, dict]:
    checkpoint_path = Path(checkpoint_path)
    ckpt = torch.load(checkpoint_path, map_location=device)

    model_cfg = ckpt.get("model_config", {})
    model = SmallCNN(
        dropout=float(model_cfg.get("dropout", 0.3)),
        use_batchnorm=bool(model_cfg.get("use_batchnorm", True)),
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device)
    model.eval()
    return model, ckpt


def predict_wav_file(
    wav_path: Path | str,
    checkpoint_path: Path | str,
    device: str = "cpu",
) -> dict:
    wav_path = Path(wav_path)
    model, ckpt = load_checkpoint_and_model(checkpoint_path, device=device)

    segment_seconds = SEGMENT_SECONDS
    image_size = int(ckpt["image_size"])
    train_mean = float(ckpt["train_mean"])
    train_std = float(ckpt["train_std"])
    threshold = float(ckpt["threshold"])
    aggregation_method = ckpt.get("aggregation_method", "mean")

    y, sr = librosa.load(wav_path, sr=None, mono=True)
    segment_len = int(round(segment_seconds * sr))

    if len(y) <= segment_len:
        segments = [_pad_or_trim_segment(y, segment_len)]
    else:
        segments = []
        for start in range(0, len(y), segment_len):
            end = start + segment_len
            seg = y[start:end]
            if len(seg) < segment_len:
                break
            segments.append(seg)

    probs = []
    with torch.no_grad():
        for seg in segments:
            arr = mel_spectrogram_uint8(seg, sr=sr, image_size=image_size)
            img = Image.fromarray(arr, mode="L")
            tmp_path = PATHS.work_dir / "__tmp_predict.png"
            img.save(tmp_path)
            x = load_png_as_tensor(tmp_path, image_size=image_size, mean=train_mean, std=train_std, augment=False)
            tmp_path.unlink(missing_ok=True)

            x = x.unsqueeze(0).to(device)
            prob = torch.sigmoid(model(x)).item()
            probs.append(float(prob))

    if aggregation_method == "mean":
        recording_prob = float(np.mean(probs))
    elif aggregation_method == "median":
        recording_prob = float(np.median(probs))
    elif aggregation_method == "top3mean":
        k = min(3, len(probs))
        recording_prob = float(np.mean(np.sort(np.array(probs))[-k:]))
    else:
        raise ValueError(f"Unknown aggregation method: {aggregation_method}")

    pred = int(recording_prob >= threshold)
    return {
        "wav_path": str(wav_path),
        "n_segments": len(probs),
        "segment_probs_allow": probs,
        "recording_prob_allow": recording_prob,
        "threshold": threshold,
        "predicted_label": pred,
        "predicted_class_name": "allow" if pred == 1 else "not_allow",
    }

In [ ]:
# Przyklad:
# checkpoint = PATHS.model_dir / f"{EXPERIMENT_NAME}.pt"
# pred = predict_wav_file(
#     wav_path=Path(r"sciezka/do/nowego_pliku.wav"),
#     checkpoint_path=checkpoint,
#     device=DEVICE,
# )
# pred